# Incident risk modeling — results

Multi-label prediction: resident-week index `t`, labels = incident types in the **next 30 days**.

Run `python run_pipeline.py` from `tricura_test/` to regenerate artifacts.


## Labeling logic

We predict **incident risk over the next 30 days** on a **resident–time panel** (not one row per resident).

**Unit of analysis.** Each row is a `(resident_id, index_date)` pair: a resident in-facility at weekly index dates from admission until discharge, death, or end of observation. The last index date is at least 30 days before that end date so every row has a full forward window.

**What each label means.** For index date `t` and each type in `LABEL_TYPES` (Fall, Wound, Altercation), the target is **1** if there is **at least one** non-strikeout incident of that type with `occurred_at` in **(t, t + 30 days]**, and **0** otherwise. Multi-label: a row can have none, one, or several types positive.

**Residents with zero incidents.** Residents who never have an incident, and weeks with no event in the forward window, are included with **all labels = 0**. There is no separate “no incident” class.

**Excluded.** Strikeout incidents; types below `MIN_EVENTS_PER_LABEL` (50); residents without `admission_date`.

**Leakage.** Labels use events **after** `t`. Features use only data **strictly before** `t`. Train/test split is **chronological** on `index_date`.

In [61]:
import json
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import plotly.express as px

ROOT = Path('..').resolve()
REPORTS = ROOT / 'reports'

with open(REPORTS / 'split_info.json') as f:
    split_info = json.load(f)
with open(REPORTS / 'metrics.json') as f:
    metrics = json.load(f)
with open(REPORTS / 'counterfactual_comparison.json') as f:
    counterfactual = json.load(f)
with open(REPORTS / 'scenario_summary.json') as f:
    scenario = json.load(f)
with open(REPORTS / 'test_cost_scenario.json') as f:
    test_cost = json.load(f)

benchmark = pd.read_csv(REPORTS / 'model_benchmark.csv')
effects = pd.read_csv(REPORTS / 'feature_effects.csv')
cf_plan = pd.read_csv(REPORTS / 'counterfactual_feature_plan.csv')
cf_selected = pd.read_csv(REPORTS / 'counterfactual_selected_features.csv')

champion = metrics.get('champion_model', metrics.get('model', 'logistic'))
print(f"Champion model (test macro recall): {champion}")
print(f"Benchmark ranking: {metrics.get('benchmark_ranking', [])}")


Champion model (test macro recall): xgboost
Benchmark ranking: ['xgboost', 'logistic', 'tpot', 'knn']


## Train, validation, and test datasets

Chronological split: **train** → **validation** (tuning / early stopping) → **test** (held out). See `split_info['description']` in the next table.


In [62]:
split_labels = {
    'train': 'Train',
    'validation': 'Validation',
    'test': 'Test (held-out)',
}
cohort_rows = []
for split in ('train', 'validation', 'test'):
    s = split_info[split].copy()
    s['split'] = split_labels[split]
    cohort_rows.append(s)
cohort_df = pd.DataFrame(cohort_rows)
print(split_info['description'])
cohort_df


Weekly resident-time panel (index every W). Chronological split: train < 2023-09-10 00:00:00, validation [2023-09-10 00:00:00, 2024-03-03 00:00:00), test >= 2024-03-03 00:00:00. Validation is used for hyperparameter tuning / early stopping; test is held out for final metrics. Labels = any incident of each type in the next 30 days.


,n_rows,n_residents,date_min,date_max,positive_rate_Fall,positive_rate_Wound,positive_rate_Altercation,positive_rate_any,split
0,65596,723,2005-02-27 00:00:00,2023-09-03 00:00:00,0.006525,0.001555,0.000777,0.008354,Train
1,16406,874,2023-09-10 00:00:00,2024-02-25 00:00:00,0.071376,0.026332,0.010118,0.101061,Validation
2,35289,1459,2024-03-03 00:00:00,2024-12-29 00:00:00,0.078863,0.027317,0.008785,0.107314,Test (held-out)


In [63]:
pd.DataFrame({
    'Setting': [
        'Prediction horizon',
        'Index frequency',
        'Train/val cutoff',
        'Val/test cutoff',
        'Label types',
    ],
    'Value': [
        f"{split_info['horizon_days']} days",
        split_info['index_freq'],
        split_info['cutoff_val_start'],
        split_info['cutoff_test_start'],
        ', '.join(split_info['label_types']),
    ],
})


,Setting,Value
0,Prediction horizon,30 days
1,Index frequency,W
2,Train/val cutoff,2023-09-10 00:00:00
3,Val/test cutoff,2024-03-03 00:00:00
4,Label types,"Fall, Wound, Altercation"


## Why precision is low (and why we prioritize recall)

- **Class imbalance:** Most resident-weeks have no incident in the next 30 days (`y_any` is positive on only ~5% of the full panel; higher on the late test slice).
- **Default threshold 0.5** plus **`class_weight="balanced"`** in logistic regression pushes the model to flag more positives → **higher recall, lower precision**.
- **Multi-label:** Subset accuracy requires every type to be correct on a row; one wrong label fails the whole row.
- **Business framing:** For liability screening, **missing a future incident (false negative)** is usually costlier than a **false alert (false positive)**. We **benchmark models by test macro recall** (mean recall across Fall, Wound, Altercation), while still reporting precision, F1, and ROC/PR AUC for context.

## Model benchmark (logistic, XGBoost, KNN, TPOT)

Four multi-label classifiers on the same chronological train / validation / test split. **Champion** = highest **test macro recall** (tie-break: macro F1, macro ROC AUC).

- **Validation** tunes logistic `C`, KNN `k`, XGBoost early stopping, and TPOT pipelines (scoring = recall).
- **KNN** uses a stratified train subsample (`KNN_MAX_TRAIN_ROWS`).
- **TPOT** runs AutoML per label (~8 min/label); can be slow on first pipeline run.

In [64]:
bench_display_cols = [
    'model', 'split', 'macro_recall', 'macro_precision', 'macro_f1',
    'macro_roc_auc', 'hamming_loss', 'subset_accuracy',
]
bench_fmt = benchmark[bench_display_cols].copy()
bench_fmt['split'] = bench_fmt['split'].map({'train': 'Train', 'test': 'Test'})
bench_fmt.round(4)

bench_test = benchmark[benchmark['split'] == 'test'].copy()
fig_bench = px.bar(
    bench_test.sort_values('macro_recall', ascending=True),
    x='macro_recall', y='model', orientation='h',
    title=f'Test macro recall by model (champion: {champion})',
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_bench.show()

# Per-label recall on test
recall_cols = [c for c in benchmark.columns if c.startswith('recall_')]
bench_recall_long = bench_test.melt(
    id_vars=['model'], value_vars=recall_cols,
    var_name='incident_type', value_name='recall',
)
bench_recall_long['incident_type'] = bench_recall_long['incident_type'].str.replace('recall_', '').str.replace('_', ' ')
fig_recall = px.bar(
    bench_recall_long, x='incident_type', y='recall', color='model', barmode='group',
    title='Test recall by incident type and model',
)
fig_recall.show()

## Champion model — quality metrics

Metrics below are for the **benchmark champion** (`logistic`, `xgboost`, or `knn` by test macro recall). Current champion: see printout in setup cell.

**Train** = in-sample (earlier weeks). **Test** = held-out (later weeks).

- **Recall** (priority): share of true incidents flagged at threshold 0.5
- **Subset accuracy**: all labels correct on a row
- **Hamming loss**: avg wrong labels per row
- **Macro F1**: mean F1 across types


In [65]:
METRIC_COLS = {
    'recall': 'Recall', 'precision': 'Precision', 'roc_auc': 'ROC AUC',
    'pr_auc': 'PR AUC', 'f1': 'F1', 'accuracy': 'Accuracy',
}

def metrics_to_long(split_name):
    block = metrics[split_name]
    rows = []
    for lt, m in block['per_label'].items():
        row = {'Incident type': lt,
               'Dataset': 'Train (in-sample)' if split_name == 'train' else 'Test (held-out)'}
        for k, label in METRIC_COLS.items():
            row[label] = m.get(k)
        rows.append(row)
    return pd.DataFrame(rows)

metrics_long = pd.concat([metrics_to_long('train'), metrics_to_long('test')], ignore_index=True)
metrics_long


,Incident type,Dataset,Recall,Precision,ROC AUC,PR AUC,F1,Accuracy
0,Fall,Train (in-sample),0.509346,0.106134,0.894892,0.080881,0.175665,0.968809
1,Wound,Train (in-sample),0.725490,0.012519,0.900744,0.024238,0.024613,0.910589
2,Altercation,Train (in-sample),0.705882,0.015471,0.938935,0.026910,0.030278,0.964845
3,Fall,Test (held-out),0.868847,0.114880,0.685723,0.128088,0.202929,0.461730
4,Wound,Test (held-out),0.893154,0.040495,0.676360,0.042587,0.077477,0.418969
5,Altercation,Test (held-out),0.748387,0.013986,0.676010,0.013704,0.027459,0.534302


In [66]:
summary_rows = []
for split_name, label in [('train', 'Train (in-sample)'), ('test', 'Test (held-out)')]:
    s = metrics[split_name]['summary']
    summary_rows.append({
        'Dataset': label,
        'Macro recall': s['macro_recall'],
        'Macro precision': s['macro_precision'],
        'Subset accuracy': s['subset_accuracy'],
        'Hamming loss': s['hamming_loss'],
        'Micro F1': s['micro_f1'],
        'Macro F1': s['macro_f1'],
        'Macro ROC AUC': s.get('macro_roc_auc'),
    })
pd.DataFrame(summary_rows)


,Dataset,Macro recall,Macro precision,Subset accuracy,Hamming loss,Micro F1,Macro F1,Macro ROC AUC
0,Train (in-sample),0.646906,0.044708,0.882584,0.051919,0.060333,0.076852,0.911524
1,Test (held-out),0.836796,0.056454,0.319051,0.528333,0.111540,0.102622,0.679364


In [67]:
plot_df = metrics_long.melt(
    id_vars=['Incident type', 'Dataset'],
    value_vars=list(METRIC_COLS.values()),
    var_name='Metric', value_name='Score',
)
fig_metrics = px.bar(
    plot_df, x='Incident type', y='Score', color='Dataset',
    facet_col='Metric', facet_col_wrap=3, barmode='group',
    title='Train vs test metrics by incident type',
)
fig_metrics.update_layout(height=500, legend=dict(orientation='h', y=-0.15))
fig_metrics.show()


## Feature drivers (logistic regression)

**Rules:** Coefficients come from **logistic regression** on standardized features (fit on train+validation), even when the benchmark champion is another model.

- **Positive coefficient** → higher feature value tends to **increase** predicted risk for that incident type.
- **Negative coefficient** → higher value tends to **decrease** risk.
- Effects rank features **across Fall, Wound, and Altercation**; counterfactuals use the top drivers below, while the **champion** model scores outcomes.


In [68]:
123

123

In [69]:
def plot_top_features(incident_type, direction, n=10):
    sub = effects.loc[(effects['incident_type']==incident_type) & (effects['direction']==direction)]
    sub = sub.nlargest(n, 'abs_coefficient')
    title_dir = 'increasing' if direction == 'increases_risk' else 'decreasing'
    fig = px.bar(sub, x='coefficient', y='feature_readable', orientation='h',
        title=f'Top {n} features {title_dir} risk — {incident_type}')
    fig.update_layout(yaxis=dict(categoryorder='total ascending'), height=420)
    fig.show()

for lt in split_info['label_types']:
    plot_top_features(lt, 'increases_risk')
    plot_top_features(lt, 'decreases_risk')


In [70]:
321

321

In [71]:
effects

,incident_type,feature,feature_readable,coefficient,direction,abs_coefficient
0,Altercation,vital_weight_last_90d,Vital (weight last 90d),0.153568,increases_risk,0.153568
1,Altercation,vital_weight_last_30d,Vital (weight last 30d),0.149224,increases_risk,0.149224
2,Altercation,days_since_admission,Days since admission,-0.148595,decreases_risk,0.148595
3,Altercation,age_years,Age at index (years),-0.132166,decreases_risk,0.132166
4,Altercation,vital_pulse_mean_90d,Vital (pulse mean 90d),0.130924,increases_risk,0.130924
...,...,...,...,...,...,...
217,Wound,gg_count_90d,Functional (GG) assessments (90d),0.000000,decreases_risk,0.000000
218,Wound,tag_assistance_needed_30d,Document tag (assistance needed 30d),0.000000,decreases_risk,0.000000
219,Wound,tag_assistance_needed_90d,Document tag (assistance needed 90d),0.000000,decreases_risk,0.000000
220,Wound,tag_antibiotic_therapy_30d,Document tag (antibiotic therapy 30d),0.000000,decreases_risk,0.000000


## Counterfactual grid search (test sample)

**What this is:** A *what-if* sensitivity check on held-out **test** rows—not proof that changing care prevents incidents.

**Feature selection (all incident types):** Top **5** logistic features that **increase** risk and top **5** that **decrease** risk.

**Percentile grid (per row, train+val reference):**
- **Increase-risk features:** from the row’s current percentile on the train distribution, step **down by 10** until **p25** (e.g. 50 → 40 → 30 → 25).
- **Decrease-risk features (inverse):** step **up by 10** until **p75**.

Each trial sets the feature to the train reference **value at that percentile**. Greedy search picks the best step per feature. Grid on **5k subsample**; implied cost on **full test**.

**Scores:** Champion model only; **no retraining**.


### Selected features (top 5 increase + top 5 decrease)

In [72]:
cf_selected

,feature,feature_readable,incident_type,coefficient,abs_coefficient,direction,grid_direction
0,dx_distinct_icd,Distinct ICD codes,Wound,1.316029,1.316029,increases_risk,increase
1,vital_count_30d,Vital sign events (30d),Fall,1.024712,1.024712,increases_risk,increase
2,tag_physician_notification_90d,Document tag (physician notification 90d),Altercation,0.585190,0.585190,increases_risk,increase
3,age_years,Age at index (years),Fall,0.534926,0.534926,increases_risk,increase
4,vital_count_90d,Vital sign events (90d),Altercation,0.503752,0.503752,increases_risk,increase
5,dx_active_count,Active diagnoses,Wound,-0.959020,0.959020,decreases_risk,decrease
6,outpatient,Outpatient flag,Altercation,-0.755061,0.755061,decreases_risk,decrease
7,days_since_admission,Days since admission,Altercation,-0.605929,0.605929,decreases_risk,decrease
8,med_on_time_rate_90d,Medication on-time rate (90d),Altercation,-0.595680,0.595680,decreases_risk,decrease
9,tag_wound_care_90d,Document tag (wound care 90d),Altercation,-0.592780,0.592780,decreases_risk,decrease


In [73]:
cf_rows = []
for lt, v in counterfactual['per_label'].items():
    cf_rows.append({
        'Incident type': lt,
        'Mean prob (baseline)': v['mean_predicted_prob_baseline'],
        'Mean prob (optimized)': v['mean_predicted_prob_optimized'],
        'Percent reduction': v['pct_reduction'],
    })
cf_df = pd.DataFrame(cf_rows)
cf_df


,Incident type,Mean prob (baseline),Mean prob (optimized),Percent reduction
0,Fall,0.495967,0.081456,83.576281
1,Wound,0.497557,0.066109,86.713273
2,Altercation,0.458320,0.079136,82.733425


In [74]:
cf_long = cf_df.melt(
    id_vars=['Incident type'],
    value_vars=['Mean prob (baseline)', 'Mean prob (optimized)'],
    var_name='Scenario', value_name='Mean predicted probability',
)
fig_cf = px.bar(cf_long, x='Incident type', y='Mean predicted probability', color='Scenario',
    barmode='group', title=f"Test n={counterfactual['n_test_sample']:,}: baseline vs optimized")
fig_cf.show()
cf_plan


,feature,grid_direction,direction,incident_type,coefficient,chosen_step,chosen_percentile,avg_value_before,avg_value_after
0,dx_distinct_icd,increase,increases_risk,Wound,1.316029,7,25.3150,13.964400,0.000000
1,vital_count_30d,increase,increases_risk,Fall,1.024712,2,59.1642,119.562000,0.000000
2,tag_physician_notification_90d,increase,increases_risk,Altercation,0.585190,1,46.0666,1.196400,0.000000
3,age_years,increase,increases_risk,Fall,0.534926,8,25.0000,75.683089,66.656400
4,vital_count_90d,increase,increases_risk,Altercation,0.503752,2,60.1520,338.163400,0.000000
5,dx_active_count,decrease,decreases_risk,Wound,-0.959020,6,75.0000,14.110200,17.000000
6,outpatient,decrease,decreases_risk,Altercation,-0.755061,0,50.1700,0.043400,0.043400
7,days_since_admission,decrease,decreases_risk,Altercation,-0.605929,8,75.0000,672.289600,1100.000000
8,med_on_time_rate_90d,decrease,decreases_risk,Altercation,-0.595680,0,54.8340,0.067383,0.074987
9,tag_wound_care_90d,decrease,decreases_risk,Altercation,-0.592780,0,55.4400,1.293800,1.196800


## Test implied cost (baseline vs optimized)

On the **full held-out test set**, implied cost = Σ (predicted probability × README average cost per incident type). **Baseline** uses the champion model on unmodified features; **optimized** applies the counterfactual feature plan from the grid search (same reductions, replayed on all test rows). Historical monthly forecast remains in `reports/scenario_summary.json`.


In [75]:
def _fmt_cost(x: float) -> str:
    if x >= 1_000_000:
        return f'${x/1e6:.2f}M'
    if x >= 1_000:
        return f'${x/1e3:.0f}K'
    return f'${x:,.0f}'

compare = pd.read_csv(REPORTS / 'test_cost_scenario_compare.csv')
baseline = test_cost['baseline_total']
optimized = test_cost['optimized_total']
savings = test_cost['savings']
savings_pct = test_cost['savings_pct']

compare['label'] = compare['expected_cost'].map(_fmt_cost)
fig_cost = px.bar(
    compare, x='scenario', y='expected_cost',
    text='label',
    title='Test implied cost: baseline vs counterfactual optimization',
)
fig_cost.update_traces(textposition='outside')
fig_cost.add_annotation(
    text=f'Save {_fmt_cost(savings)} ({savings_pct:.1f}%)',
    xref='paper', yref='paper', x=0.5, y=1.08, showarrow=False,
    font=dict(size=14),
)
fig_cost.update_layout(yaxis_title='Implied cost (USD)', uniformtext_minsize=10)
fig_cost.show()

